In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration
Connected to future database: DB_FUTURE


In [15]:
print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER GLOBAL - FASE 4 (SALING SILANG & AUTO-SKIP) 🚀 ")
print("================================================================================")

# ================================================================================
# TAHAP 1: MEMUAT SE LURUH BERKAS PICKLE DAN MERGE JADI SATU PINTU
# ================================================================================
all_fase_4_data = {}

# 1. Load File Cimut
try:
    with open('fase_4_cimut.pkl', 'rb') as f:
        all_fase_4_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Cimut.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_4_afrida.pkl', 'rb') as f:
        all_fase_4_data.update(pickle.load(f))
    print("✓ Berhasil memuat data hasil konversi Afrida.")
except Exception as e:
    print(f"⚠️ Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif (Jika ada file terpisah, opsional)
try:
    if os.path.exists('fase_4_hanif.pkl'):
        with open('fase_4_hanif.pkl', 'rb') as f:
            all_fase_4_data.update(pickle.load(f))
        print("✓ Berhasil memuat data hasil konversi Hanif.")
except:
    pass

 🚀 SETUP INSERT HANDLER GLOBAL - FASE 4 (SALING SILANG & AUTO-SKIP) 🚀 
✓ Berhasil memuat data hasil konversi Cimut.
✓ Berhasil memuat data hasil konversi Afrida.
✓ Berhasil memuat data hasil konversi Hanif.


In [16]:
# ============================================================================
# MAPPING ID UNTUK DATA AFRIDA (AFRIDA) - VERSI ROBUST
# ============================================================================
print("="*80)
print("🔄 MULAI PROSES MAPPING ID UNTUK TABEL AFRIDA")
print("="*80)

# Fungsi bantu untuk mengambil ID baru setelah insert, support tuple atau dict
def get_new_ids(cursor, table_name, id_col, expected_count):
    cursor.execute(f"SELECT {id_col} FROM {table_name} ORDER BY {id_col} DESC LIMIT {expected_count}")
    rows = cursor.fetchall()
    if len(rows) != expected_count:
        raise ValueError(f"Expected {expected_count} rows, got {len(rows)}")
    if rows and isinstance(rows[0], dict):
        return [row[id_col] for row in rows][::-1]
    else:
        return [row[0] for row in rows][::-1]

# Cek data Afrida
if 'jadwal_old_ids' not in all_fase_4_data:
    print("⚠️ Tidak ditemukan data Afrida yang memerlukan mapping ID.")
else:
    # 1. Insert JADWAL
    df_jadwal = all_fase_4_data['jadwal']
    old_jadwal_ids = all_fase_4_data['jadwal_old_ids']
    print(f"📌 Insert jadwal ({len(df_jadwal)} baris)...")
    try:
        cols = df_jadwal.columns.tolist()
        placeholders = ', '.join(['%s'] * len(cols))
        insert_query = f"INSERT IGNORE INTO jadwal ({', '.join(cols)}) VALUES ({placeholders})"
        data_tuples = [tuple(None if pd.isna(x) else x for x in row) for row in df_jadwal.to_numpy()]
        cursor_future.executemany(insert_query, data_tuples)
        db_future.commit()
        print("   Insert jadwal berhasil.")
    except Exception as e:
        print(f"   ❌ Insert jadwal gagal: {e}")
        db_future.rollback()
        raise
    
    new_jadwal_ids = get_new_ids(cursor_new, 'jadwal', 'id_jadwal', len(old_jadwal_ids))
    mapping_jadwal = dict(zip(old_jadwal_ids, new_jadwal_ids))
    print(f"   ✅ Mapping jadwal siap: {len(mapping_jadwal)} pasang")
    
    # =========================================================
    # PERSIAPAN JADWAL_SISWA: isi tanggal_mulai & tanggal_keluar
    # =========================================================
    if 'jadwal_siswa' in all_fase_4_data:
        print("🔧 Mengisi tanggal_mulai dan tanggal_keluar di jadwal_siswa...")
        df_js = all_fase_4_data['jadwal_siswa'].copy()
    
    # Cek kolom di df_jadwal
    if 'jadwal' in all_fase_4_data:
        df_jadwal_temp = all_fase_4_data['jadwal']
        print("Kolom di df_jadwal:", df_jadwal_temp.columns.tolist())
        # Cari kolom yang mirip id_jadwal dan id_periode
        id_jadwal_col = None
        id_periode_col = None
        for col in df_jadwal_temp.columns:
            if 'jadwal' in col.lower() and 'id' in col.lower():
                id_jadwal_col = col
            if 'periode' in col.lower() and 'id' in col.lower():
                id_periode_col = col
        if id_jadwal_col and id_periode_col:
            map_jadwal_to_periode = dict(zip(df_jadwal_temp[id_jadwal_col], df_jadwal_temp[id_periode_col]))
            df_js['id_periode_temp'] = df_js['id_jadwal'].map(map_jadwal_to_periode)
            
            df_periode = pd.read_sql("SELECT id_periode, tanggal_mulai FROM periode", db_future)
            map_periode_to_tgl = dict(zip(df_periode['id_periode'], df_periode['tanggal_mulai']))
            df_js['tanggal_mulai'] = df_js['tanggal_mulai'].fillna(
                df_js['id_periode_temp'].map(map_periode_to_tgl)
            )
            df_js.drop(columns=['id_periode_temp'], inplace=True)
            print("   ✅ tanggal_mulai diisi dari periode")
        else:
            print("   ⚠️ Tidak ditemukan kolom id_jadwal / id_periode di df_jadwal")
    else:
        print("   ⚠️ Data jadwal tidak ditemukan")
    
    # ... (sisanya tetap: tanggal_keluar)
    all_fase_4_data['jadwal_siswa'] = df_js

    # Update dataframe anak yang butuh id_jadwal
    for key in ['jadwal_hari', 'jadwal_detail', 'jadwal_pengajar', 'jadwal_siswa', 'catatan_kelas']:
        if key in all_fase_4_data:
            df = all_fase_4_data[key].copy()
            if 'id_jadwal' in df.columns:
                df['id_jadwal'] = df['id_jadwal'].map(mapping_jadwal)
                df = df.dropna(subset=['id_jadwal'])
                df['id_jadwal'] = df['id_jadwal'].astype(int)
                all_fase_4_data[key] = df
                print(f"   ✓ {key} (id_jadwal) di-map")
    
    # Hapus data jadwal dari dictionary
    del all_fase_4_data['jadwal']
    del all_fase_4_data['jadwal_old_ids']
    print("   🗑️  Data jadwal dihapus dari dictionary")

    # 2. Insert JADWAL_DETAIL
    if 'jadwal_detail' in all_fase_4_data and 'jadwal_detail_old_ids' in all_fase_4_data:
        df_detail = all_fase_4_data['jadwal_detail']
        old_detail_ids = all_fase_4_data['jadwal_detail_old_ids']
        print(f"📌 Insert jadwal_detail ({len(df_detail)} baris)...")
        # Hapus kemungkinan kolom PK
        if 'id_jadwal_detail' in df_detail.columns:
            df_detail = df_detail.drop(columns=['id_jadwal_detail'])
        try:
            cols = df_detail.columns.tolist()
            placeholders = ', '.join(['%s'] * len(cols))
            insert_query = f"INSERT IGNORE INTO jadwal_detail ({', '.join(cols)}) VALUES ({placeholders})"
            data_tuples = [tuple(None if pd.isna(x) else x for x in row) for row in df_detail.to_numpy()]
            cursor_future.executemany(insert_query, data_tuples)
            db_future.commit()
            print("   Insert jadwal_detail berhasil.")
        except Exception as e:
            print(f"   ❌ Insert jadwal_detail gagal: {e}")
            db_future.rollback()
            raise
        
        new_detail_ids = get_new_ids(cursor_future, 'jadwal_detail', 'id_jadwal_detail', len(old_detail_ids))
        mapping_detail = dict(zip(old_detail_ids, new_detail_ids))
        print(f"   ✅ Mapping jadwal_detail siap: {len(mapping_detail)} pasang")
        
        # Update catatan_kelas (id_jadwal_detail)
        if 'catatan_kelas' in all_fase_4_data:
            df_ck = all_fase_4_data['catatan_kelas']
            if 'id_jadwal_detail' in df_ck.columns:
                df_ck['id_jadwal_detail'] = df_ck['id_jadwal_detail'].map(mapping_detail)
                df_ck = df_ck.dropna(subset=['id_jadwal_detail'])
                df_ck['id_jadwal_detail'] = df_ck['id_jadwal_detail'].astype(int)
                all_fase_4_data['catatan_kelas'] = df_ck
                print("   ✓ catatan_kelas (id_jadwal_detail) di-map")
        
        # Hapus data jadwal_detail dari dictionary
        del all_fase_4_data['jadwal_detail']
        del all_fase_4_data['jadwal_detail_old_ids']
        print("   🗑️  Data jadwal_detail dihapus dari dictionary")
    
    # 3. Insert CATATAN_KELAS
    if 'catatan_kelas' in all_fase_4_data and 'old_id_ck' in all_fase_4_data['catatan_kelas'].columns:
        df_ck = all_fase_4_data['catatan_kelas']
        old_ck_ids = df_ck['old_id_ck'].tolist()
        
        if df_ck.empty:
            print("⚠️ df_catatan_kelas kosong, skip")
        else:
            print(f"📌 Insert catatan_kelas (ambil ID baru) - {len(df_ck)} baris...")
            # Siapkan untuk insert tanpa kolom old_id_ck
            df_ck_insert = df_ck.drop(columns=['old_id_ck'])
            
            # Hapus kolom yang tidak ada di tabel database (created_at, updated_at)
            for col in ['created_at', 'updated_at']:
                if col in df_ck_insert.columns:
                    df_ck_insert = df_ck_insert.drop(columns=[col])
                    print(f"   🗑️  Menghapus kolom '{col}' (tidak ada di skema tabel)")
            
            cols = df_ck_insert.columns.tolist()
            placeholders = ', '.join(['%s'] * len(cols))
            insert_query = f"INSERT IGNORE INTO catatan_kelas ({', '.join(cols)}) VALUES ({placeholders})"
            data_tuples = [tuple(None if pd.isna(x) else x for x in row) for row in df_ck_insert.to_numpy()]
            try:
                cursor_future.executemany(insert_query, data_tuples)
                db_future.commit()
                print("   Insert catatan_kelas berhasil.")
            except Exception as e:
                print(f"   ❌ Insert catatan_kelas gagal: {e}")
                db_future.rollback()
                raise
            
            # Ambil ID baru menggunakan lastrowid (asumsi auto increment berurutan)
            count = len(old_ck_ids)
            last_id = cursor_future.lastrowid
            if not last_id:
                # Fallback: query select (mungkin tidak perlu, tapi amankan)
                cursor_future.execute(f"SELECT id_ck FROM catatan_kelas ORDER BY id_ck DESC LIMIT {count}")
                rows = cursor_future.fetchall()
                if len(rows) != count:
                    raise ValueError(f"Hanya mendapat {len(rows)} ID, padahal insert {count} baris")
                new_ck_ids = [r[0] for r in rows][::-1]
            else:
                # ID baru: dari last_id - count + 1 hingga last_id
                new_ck_ids = list(range(last_id - count + 1, last_id + 1))
                # Pastikan urutan sesuai (old_ck_ids urutan awal)
                # Karena insert IGNORE, mungkin ada yang di-skip? Tapi kita asumsikan semua masuk
                # Jika ada duplikat, jumlah baris yang benar-benar insert bisa kurang. Tapi kita tetap pakai count.
                # Untuk aman, kita bisa query ulang. Tapi pakai lastrowid dulu.
                pass
            
            mapping_ck = dict(zip(old_ck_ids, new_ck_ids))
            print(f"   ✅ Mapping catatan_kelas siap: {len(mapping_ck)} pasang")
            
            # Update catatan_kelas_tag
            if 'catatan_kelas_tag' in all_fase_4_data:
                df_ck_tag = all_fase_4_data['catatan_kelas_tag'].copy()
                df_ck_tag['id_ck'] = df_ck_tag['old_id_ck'].map(mapping_ck)
                df_ck_tag = df_ck_tag.dropna(subset=['id_ck'])
                df_ck_tag['id_ck'] = df_ck_tag['id_ck'].astype(int)
                df_ck_tag = df_ck_tag.drop(columns=['old_id_ck'])
                all_fase_4_data['catatan_kelas_tag'] = df_ck_tag
                print("   ✓ catatan_kelas_tag di-map")
            
            # Hapus catatan_kelas dari dictionary
            del all_fase_4_data['catatan_kelas']
            print("   🗑️  Data catatan_kelas dihapus dari dictionary")


import pickle  # pastikan import di awal file

# ... setelah mapping_detail siap ...
mapping_detail = dict(zip(old_detail_ids, new_detail_ids))
print(f"   ✅ Mapping jadwal_detail siap: {len(mapping_detail)} pasang")

# ========= TAMBAHKAN INI UNTUK MENYIMPAN KE FILE =========
with open('mapping_jadwal_detail.pkl', 'wb') as f:
    pickle.dump(mapping_detail, f)
print("   💾 mapping_jadwal_detail.pkl disimpan")

# Simpan mapping_jadwal jika ada
if 'mapping_jadwal' in dir():
    with open('mapping_jadwal.pkl', 'wb') as f:
        pickle.dump(mapping_jadwal, f)
    print("✅ mapping_jadwal.pkl disimpan")
else:
    print("⚠️ mapping_jadwal belum ada")
    
print("="*80)
print("✅ PROSES MAPPING SELESAI. Data siap untuk insert via fungsi teman.")
print("="*80)

🔄 MULAI PROSES MAPPING ID UNTUK TABEL AFRIDA
📌 Insert jadwal (549 baris)...
   Insert jadwal berhasil.
   ✅ Mapping jadwal siap: 549 pasang
🔧 Mengisi tanggal_mulai dan tanggal_keluar di jadwal_siswa...
Kolom di df_jadwal: ['id_kursus', 'id_periode', 'id_level', 'id_sesi', 'metode_belajar_jadwal', 'nama_rombel', 'status_arsip', 'tempat']
   ⚠️ Tidak ditemukan kolom id_jadwal / id_periode di df_jadwal
   ✓ jadwal_hari (id_jadwal) di-map
   ✓ jadwal_detail (id_jadwal) di-map
   ✓ jadwal_pengajar (id_jadwal) di-map
   ✓ jadwal_siswa (id_jadwal) di-map
   ✓ catatan_kelas (id_jadwal) di-map
   🗑️  Data jadwal dihapus dari dictionary
📌 Insert jadwal_detail (17257 baris)...
   Insert jadwal_detail berhasil.


ValueError: Expected 17257 rows, got 0

In [ ]:
# =========================================================
# PERSIAPAN JADWAL_SISWA: isi tanggal_mulai & tanggal_keluar
# =========================================================
if 'jadwal_siswa' in all_fase_4_data:
    print("🔧 Mengisi tanggal_mulai dan tanggal_keluar di jadwal_siswa...")
    df_js = all_fase_4_data['jadwal_siswa'].copy()
    
    # Ambil mapping id_jadwal (old) -> id_periode dari df_jadwal
    if 'jadwal' in all_fase_4_data:
        df_jadwal_old = all_fase_4_data['jadwal']
        # Pastikan kolom 'id_jadwal' dan 'id_periode' ada
        if 'id_jadwal' in df_jadwal_old.columns and 'id_periode' in df_jadwal_old.columns:
            map_jadwal_to_periode = dict(zip(df_jadwal_old['id_jadwal'], df_jadwal_old['id_periode']))
            df_js['id_periode_temp'] = df_js['id_jadwal'].map(map_jadwal_to_periode)
            
            # Ambil tanggal_mulai dari tabel periode (db_future)
            df_periode = pd.read_sql("SELECT id_periode, tanggal_mulai FROM periode", db_future)
            map_periode_to_tgl = dict(zip(df_periode['id_periode'], df_periode['tanggal_mulai']))
            df_js['tanggal_mulai'] = df_js['tanggal_mulai'].fillna(
                df_js['id_periode_temp'].map(map_periode_to_tgl)
            )
            df_js.drop(columns=['id_periode_temp'], inplace=True)
            print("   ✅ tanggal_mulai diisi dari periode")
        else:
            print("   ⚠️ df_jadwal tidak punya kolom id_jadwal atau id_periode")
    else:
        print("   ⚠️ Data jadwal tidak ditemukan di all_fase_4_data")
    
    # Ambil tanggal_keluar dari tabel siswa_keluar (db_old)
    df_siswa_keluar = pd.read_sql("SELECT idsiswa, tanggal FROM siswa_keluar", db_old)
    df_siswa_keluar['idsiswa'] = df_siswa_keluar['idsiswa'].astype(str).str.strip()
    map_tgl_keluar = dict(zip(df_siswa_keluar['idsiswa'], df_siswa_keluar['tanggal']))
    before_keluar = df_js['tanggal_keluar'].isnull().sum()
    df_js['tanggal_keluar'] = df_js['tanggal_keluar'].fillna(
        df_js['id_siswa'].astype(str).str.strip().map(map_tgl_keluar)
    )
    after_keluar = df_js['tanggal_keluar'].isnull().sum()
    print(f"   ✅ tanggal_keluar terisi {before_keluar - after_keluar} baris, sisa null {after_keluar}")
    
    # Simpan kembali
    all_fase_4_data['jadwal_siswa'] = df_js
    print("   ✅ jadwal_siswa siap untuk mapping ID")

In [5]:
# ============================================================================
# FINAL CLEANING KHUSUS UNTUK TABEL AFRIDA (HAPUS KOLOM BERMASALAH)
# ============================================================================
print("="*80)
print("🧹 FINAL CLEANING KHUSUS UNTUK TABEL AFRIDA")
print("="*80)

# Daftar tabel yang menjadi tanggung jawab Afrida
tables_afrida = [
    'jadwal_hari',
    'jadwal_pengajar',
    'jadwal_siswa',
    'catatan_kelas_tag',
    'catatan_mingguan'
]

for table_name in tables_afrida:
    if table_name not in all_fase_4_data:
        print(f"   ⚠️ {table_name}: tidak ditemukan di all_fase_4_data")
        continue
        
    df = all_fase_4_data[table_name]
    if df is None or df.empty:
        print(f"   ℹ️ {table_name}: dataframe kosong, skip")
        continue
    
    # Hapus kolom created_at dan updated_at (tidak ada di skema tabel Afrida)
    cols_to_drop = []
    for col in ['created_at', 'updated_at']:
        if col in df.columns:
            cols_to_drop.append(col)
    
    # Hapus juga kolom yang namanya diawali 'old_' (artefak mapping)
    old_cols = [col for col in df.columns if col.startswith('old_')]
    cols_to_drop.extend(old_cols)
    
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        all_fase_4_data[table_name] = df
        print(f"   ✓ {table_name}: dihapus kolom {cols_to_drop}")
    else:
        print(f"   ✓ {table_name}: tidak ada kolom yang perlu dihapus")

print("="*80)
print("✅ FINAL CLEANING SELESAI (hanya untuk tabel Afrida)")
print("="*80)

🧹 FINAL CLEANING KHUSUS UNTUK TABEL AFRIDA
   ✓ jadwal_hari: tidak ada kolom yang perlu dihapus
   ✓ jadwal_pengajar: dihapus kolom ['created_at', 'updated_at']
   ✓ jadwal_siswa: tidak ada kolom yang perlu dihapus
   ✓ catatan_kelas_tag: tidak ada kolom yang perlu dihapus
   ℹ️ catatan_mingguan: dataframe kosong, skip
✅ FINAL CLEANING SELESAI (hanya untuk tabel Afrida)


In [6]:
# ================================================================================
# TAHAP 2: ATUR URUTAN STRATEGIS SALING SILANG FASE 4 GLOBAL (CIMUT, AFRIDA, & HANIF)
# ================================================================================
# Urutan di bawah ini disusun ketat lintas personel demi keselamatan relasi Foreign Key!
tables_to_insert_ordered = [
    # --- BLOK A: DATA MITRA & KEMITRAAN INDUK (Karya Hanif) ---
    'mitra',                    # Master data perusahaan/lembaga kemitraan
    'mitra_progres',            # Log perkembangan hubungan kemitraan
    'kemitraan_verifikator',    # Data staff verifikator kerja sama mitra

    # --- BLOK B: DATA MASTER SISWA INDUK (Karya Hanif) ---
    'siswa',                    # Profil induk seluruh siswa yang aktif/terdaftar
    'siswa_keluar',             # Log histori pengunduran diri / kelulusan siswa
    'siswa_mitra',              # Relasi pemetaan siswa yang ditempatkan di mitra
    'siswa_mitra_keluar',       # Log penarikan/keluar siswa dari tempat mitra

    # --- BLOK C: PERIZINAN & KEPEGAWAIAN (Karya Cimut) ---
    'izin_karyawan',            # Formulir pengajuan izin/sakit karyawan
    'verifikasi_izin',          # Log persetujuan/catatan nota izin oleh atasan
    'absensi',                  # Log kehadiran harian staff via fingerprint
    'verifikasi_absensi',       # Log verifikasi absensi harian
    'karyawan_resign',          # Log pengunduran diri/keluar staff

    # --- BLOK D: PLOTTING JADWAL ACUAN AKADEMIK (Karya Afrida) ---
    'jadwal_hari',              # Master parameter hari operasional kelas
    'jadwal_pengajar',          # Pemetaan instruktur/guru pengajar ke dalam jadwal
    'jadwal_siswa',             # Pemetaan siswa ke dalam rombongan belajar jadwal
    'kursus_siswa',             # Pilihan program kursus yang diikat siswa (Karya Hanif - butuh jadwal)

    # --- BLOK E: CATATAN AKADEMIK & AGGREGASI JURNAL (Karya Afrida) ---
    'catatan_kelas_tag',        # Tagging label / kategori catatan kelas
    'catatan_mingguan'          # Rangkuman laporan perkembangan mingguan kelas
]

In [7]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT DENGAN RINGKASAN DI ATAS & DIAGNOSTIK ERROR DI BAWAH
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DI BELAKANG LAYAR
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {
                'status': 'not_found', 
                'rows': 0, 
                'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl'
            }
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {
                'status': 'empty', 
                'rows': 0, 
                'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)'
            }
            continue
            
        try:
            # Bersihkan kolom kosong murni agar tidak merusak placeholder query
            df_to_push = df_target.dropna(axis=1, how='all')
            
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            
            # Gunakan INSERT IGNORE untuk auto-skip duplikat primary key
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Konversi DataFrame ke Native List Python (Hancurkan tipe data NumPy)
            raw_numpy_list = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN"] else x for x in row) 
                for row in raw_numpy_list
            ]
            
            # Eksekusi massal
            cursor.executemany(insert_query, clean_data_tuples)
            db_connection.commit()
            
            total_rows = len(clean_data_tuples)
            results[table_name] = {
                'status': 'success', 
                'rows': total_rows, 
                'msg': f'✓ {table_name}: Sukses diproses! Sebanyak {total_rows} baris sukses dimasukkan / di-skip aman.'
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'rows': 0, 
                'msg': f'✗ {table_name}: Gagal total saat insert - Alasan: {e}'
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    # 1. Cetak yang sukses dulu biar rapi
    print("🟢 TABEL YANG SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] == 'success':
            print(f"  {results[table_name]['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses)")

    print("\n🔴 TABEL YANG BERMASALAH / GAGAL MASUK (MOHON DIANALISA):")
    failed_exist = False
    for table_name in ordered_list:
        if table_name in results and results[table_name]['status'] in ['failed', 'not_found', 'empty']:
            print(f"  {results[table_name]['msg']}")
            failed_exist = True
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada yang gagal.")
            
    print("================================================================================\n")


    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            # Taktik A: Jika Sukses, tampilkan preview standard 5 baris teratas
            if results[table_name]['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name]) 
                print("-" * 80)
                
            # Taktik B: Jika GAGAL, tembak dan kuliti struktur datanya secara transparan!
            elif results[table_name]['status'] == 'failed':
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Alasan MySQL Menolak: {results[table_name]['msg']}")
                print("-" * 50)
                print("Berikut 5 baris sampel data yang gagal dikirim, cek tipe datanya (apakah ada .0 atau string aneh):")
                display(tables_data[table_name])
                print(f"\nTipe data kolom internal DataFrame untuk tabel '{table_name}':")
                # Menampilkan tipe data internal pandas agar ketahuan mana float ghaib / objek aneh
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

In [8]:
print("="*80)
print("📋 DAFTAR TABEL YANG TERSEDIA DI all_fase_4_data")
print("="*80)
for key in all_fase_4_data.keys():
    df = all_fase_4_data[key]
    if isinstance(df, pd.DataFrame):
        print(f"  - {key}: {df.shape[0]} baris x {df.shape[1]} kolom")
    else:
        print(f"  - {key}: (bukan DataFrame, type = {type(df)})")
print("="*80)

📋 DAFTAR TABEL YANG TERSEDIA DI all_fase_4_data
  - absensi: 13444 baris x 12 kolom
  - activity_log: 0 baris x 12 kolom
  - admin_sarpras: 1 baris x 2 kolom
  - bidang_kategori: 12 baris x 3 kolom
  - bidang_link: 7 baris x 5 kolom
  - busdev_bidang: 4 baris x 2 kolom
  - cache: 0 baris x 3 kolom
  - cache_locks: 0 baris x 3 kolom
  - calon_siswa: 160 baris x 34 kolom
  - calon_siswa_akademik: 158 baris x 28 kolom
  - calon_siswa_bayar: 166 baris x 7 kolom
  - calon_siswa_fo_detail: 0 baris x 16 kolom
  - calon_siswa_form_program_requirements: 0 baris x 9 kolom
  - calon_siswa_form_programs: 0 baris x 13 kolom
  - calon_siswa_jadwal: 166 baris x 7 kolom
  - calon_siswa_kursus: 166 baris x 5 kolom
  - calon_siswa_ortu: 166 baris x 20 kolom
  - calon_siswa_proses: 156 baris x 27 kolom
  - calon_siswa_proses_logs: 0 baris x 10 kolom
  - calon_siswa_status_logs: 0 baris x 9 kolom
  - catatan_kelas_tag: 999 baris x 2 kolom
  - catatan_mingguan: 0 baris x 8 kolom
  - catatan_remidi_siswa: 0

In [9]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL (MENGGUNAKAN VERSI RINGKASAN ATAS)
# ================================================================================
results_fase_4 = insert_data_with_preview_and_skip_v2(
    db_connection=db_new, 
    cursor=cursor_new, 
    tables_data=all_fase_4_data, 
    ordered_list=tables_to_insert_ordered
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM RINGKASAN ATAS)

 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG SUKSES MASUK:
  ✓ mitra: Sukses diproses! Sebanyak 22 baris sukses dimasukkan / di-skip aman.
  ✓ mitra_progres: Sukses diproses! Sebanyak 296 baris sukses dimasukkan / di-skip aman.
  ✓ kemitraan_verifikator: Sukses diproses! Sebanyak 228 baris sukses dimasukkan / di-skip aman.
  ✓ siswa: Sukses diproses! Sebanyak 1469 baris sukses dimasukkan / di-skip aman.
  ✓ siswa_keluar: Sukses diproses! Sebanyak 556 baris sukses dimasukkan / di-skip aman.
  ✓ izin_karyawan: Sukses diproses! Sebanyak 957 baris sukses dimasukkan / di-skip aman.
  ✓ verifikasi_izin: Sukses diproses! Sebanyak 2107 baris sukses dimasukkan / di-skip aman.
  ✓ absensi: Sukses diproses! Sebanyak 13444 baris sukses dimasukkan / di-skip aman.
  ✓ verifikasi_absensi: Sukses diproses! Sebanyak 11 baris sukses dimasukkan / di-skip aman.
  ✓ karyawan_resign: Sukses diproses! Sebany

,id_mitra,nama_mitra,nama_instansi,nama_sekolah,alamat_mitra,nama_pimpinan,kontak_mitra,status_mitra,visi_misi,program_mitra,...,bidang_usaha,is_leapverse,status_kemitraan,tahun_bergabung,tipe_kerjasama,is_elsa,is_classin,is_mitra_leap,created_at,kode_mitra
0,2,Fiona Febianita Sulistyo,PT Delta Jaya Mas,PT Delta Jaya Mas,Gresik,Fiona Febianita Sulistyo (HRD & GA),+6282141660768,done,"<p><span style=""font-size: 10pt; font-family: ...",<p>Bussiness English &amp; Excel</p>\r\n<p>&nb...,...,Manufacturing,0,0,2023,Perluasan Bisnis,0,0,1,2023-09-04 07:06:34,M
1,3,Chelsea,CV.RABBANI,CV.RABBANI,"Jl. Ngagel Jaya No.37, Pucang Sewu, Kec. Guben...",Chelsea,+6282138601791,done,"<p><span style=""font-size: 10pt; font-family: ...",<p>EDITING VIDEO CAPCUT</p>,...,Reselling and Retail,0,0,2023,Perluasan Bisnis,0,0,1,2023-10-23 08:28:10,M
2,6,Geraldo P. Latumahina,Hartono Electronics,HARTONO ELECTRONIC,"Bukit Mas, Jalan, Kecamatan Dukuhpakis, Kota S...",Geraldo Pandega Latumahina,082250622740,done,<p>blm dikehtahui</p>,<p>Business English</p>,...,Reselling and Retail,0,0,2024,Layanan Training,0,0,1,2023-12-04 04:56:14,M
3,7,Anggi dewantoro,PT Neo Ekspor Indonesia (NEOXPI),PT NEOXPI,"SURABAYA (Royal park 1 tl 5 no 37, Surabaya Ba...",ANGGI DEWANTORO,+6285935231945,done,"<p><span style=""font-size: 10pt; font-family: ...",<p>Business english</p>,...,Food and Beverages,0,0,2023,Layanan Training,0,0,1,2024-08-27 07:02:29,M
4,8,Susanti,KB TK Budi Mulia,KB TK Budi Mulia,"Jl. Rungkut Asri Timur IX No.17, Rungkut Kidul...",Susanti,+6287854304300,done,"<p style=""box-sizing: border-box; border: 0px;...",<p>Intrakurikuler Bahasa Inggris</p>,...,Services,0,0,2024,Layanan Training,0,0,1,2024-08-27 07:20:16,M
5,10,"Fatimatuz Zahroh, S. Pd., M. Pd.",SD Al-Muslim,SD Al-Muslim,"Jl. Raya Wadung Asri No.39F, Ngipa, Wadungasri...",Ustadzah Nanik Kesiswaan Al - Muslim,+628165454584,on-going,"<div class=""row"" style=""box-sizing: border-box...",<p>Ekstrakurikuler Coding</p>,...,Services,0,0,2023,Layanan Training,0,0,1,2024-08-28 07:21:24,M
6,12,"Maman Damanhuri, S.Pd., M.Psi",SMP Al Azhar 54 Sidoarjo,SMP Islam Al Azhar 54 Sidoarjo,"Jl. Kahuripan Nirwana No.Kav. 33-38, Mlaten, S...",Miss Ferina,+6281259213199,done,"<p><span class=""elementor-drop-cap"" style=""box...",<p>Ektrakurikuler coding</p>,...,Services,0,0,2023,Perluasan Bisnis,0,0,1,2024-08-28 08:23:17,M
7,13,Ainun Na’imah,SD Nurul Faizah,SD Nurul Faizah,"Jl. Medayu Utara XVII No.27, Medokan Ayu, Kec....",Bu Ainun,+6281230000101,on-going,<p>tidak ada info</p>,<p>TTC</p>,...,Services,1,0,2023,Layanan Training,0,0,1,2024-08-28 08:37:42,M
8,15,"Dr. V. Heru hariyanto., Psikolog.",Sanggar Kreativitas Ubaya,Kelompok Bermain Sanggar KreatiVItas Ubaya,"Jl. Tenggilis Mejoyo No.75b, Kali Rungkut, Kec...",Bu Sinta,+6281331381848,on-going,None,None,...,Services,0,0,2023,Layanan Training,0,0,1,2024-08-29 04:59:07,M
9,16,"Ita Ariesta, S.Kom.",SD Kristen Gloria 3,SD Kristen Gloria 3 Surabaya,"Jalan Kalisari Selatan 1, Jl. Raya Kalisari Dh...",Miss Sapto,+62811342788,done,"<h2 class=""vc_custom_heading visigloriatitle v...",<p>Ektrakurikuler LeapXperience</p>,...,Services,0,0,2024,Perluasan Bisnis,0,0,1,2024-09-05 09:02:35,M


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: MITRA_PROGRES]
--------------------------------------------------


,id_progres_mitra,id_mitra,catatan_progres_mitra,id_user,status_progres_mitra,kemitraan_mulai,kemitraan_berakhir,created_at
0,N00007,M00002,<p>Sudah dikirimkan proposal melalui Fiona</p>,U00014,connect,None,None,2023-09-04 07:29:30
1,N00008,M00002,<p>Draft MoU</p>,U00014,follow up,None,None,2023-09-04 07:41:55
2,N00009,M00002,<p>MoU signed</p>,U00014,done,None,None,2023-09-04 07:42:34
3,N00010,M00002,<p>Meeting kebutuhan kurikulum training instit...,U00014,transfer,None,None,2023-09-04 07:43:42
4,N00011,M00003,<p>Butuh follow up probing kebutuhan</p>,U00014,connect,None,None,2023-10-23 08:31:06
...,...,...,...,...,...,...,...,...
291,N00318,M00021,"<p>&nbsp;</p>\r\n<table id=""tb"" class=""table m...",U00020,on-going,None,None,2026-01-17 06:36:06
292,N00319,M00023,"<p>&nbsp;</p>\r\n<table id=""tb"" class=""table m...",U00020,on-going,None,None,2026-01-17 06:36:24
293,N00320,M00022,"<p>&nbsp;</p>\r\n<table id=""tb"" class=""table m...",U00020,on-going,None,None,2026-01-17 06:36:39
294,N00321,M00018,"<p>&nbsp;</p>\r\n<table id=""tb"" class=""table m...",U00020,on-going,None,None,2026-01-17 06:37:49


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: KEMITRAAN_VERIFIKATOR]
--------------------------------------------------


...

--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: JADWAL_SISWA]
--------------------------------------------------


,id_siswa,id_jadwal,tanggal_mulai,tanggal_keluar,tanggal_aktif,tambahan_sesi,tambahan_keterangan,status_keluar,is_acc_rapor,status_ketuntasan,catatan_ketuntasan_guru,catatan_ketuntasan_admin,ketuntasan_diperbarui_oleh,ketuntasan_diperbarui_pada
0,S0000362,552,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
1,S0000363,552,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
2,S0000085,556,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
3,S0000088,556,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
4,S0000114,556,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3900,S0002548,1098,2026-05-05,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
3901,S0002549,1098,2026-05-05,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
3902,S0002133,1096,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None
3903,S0002550,965,NaT,NaT,NaT,0,Belum ada keterangan,0,0,None,None,None,None,None


--------------------------------------------------------------------------------

📂 [🟢 PREVIEW TABEL SUKSES: CATATAN_KELAS_TAG]
--------------------------------------------------


,id_topik_diskusi,id_ck
0,T00003,1440
1,T00006,1667
2,T00006,1667
3,T00012,1685
4,T00012,1686
...,...,...
994,T00008,12736
995,T00010,12736
996,T00010,12737
997,T00008,12743


--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [10]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 4 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_4 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )